In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        _tok = ""
        try:
            from google.colab import userdata
            _tok = userdata.get("GH_TOKEN") or ""
        except Exception:
            _tok = ""
        if not _tok:
            print("WARNING: no 'GH_TOKEN' Colab secret found; cloning this PRIVATE repo will fail.\n"
                  "Add a GitHub token (repo scope) via the key icon (Secrets) as 'GH_TOKEN', then re-run.")
        _url = (f"https://{_tok}@github.com/{_slug}.git" if _tok
                else f"https://github.com/{_slug}.git")
        subprocess.run(["git", "clone", "--depth", "1", _url, str(_root)], check=True)
        subprocess.run(["git", "-C", str(_root), "remote", "set-url", "origin",
                        f"https://github.com/{_slug}.git"])  # keep the token out of the saved remote
    os.chdir(_root / "06-gateway/scaling-admission-cost/agentic-scaling-lab/notebooks")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 03 · Rate limits, retries, breakers and admission control

**What you'll learn.** What a 429 means on a shared model pool, why you smooth your own traffic, why retries
need jitter, what a circuit breaker buys, and how admission control with degrade levels breaks the overload
feedback loop. Code: `scalelab/resilience.py` and `scalelab/admission.py` (about 200 lines together).

> **The one-minute version.** *"A 429 is contention on a pool I share with the whole org, not a quota I hit. I smooth my
> traffic with a token bucket sized to my share, retry with full jitter inside the turn's deadline, break the
> circuit to a sibling model, and — before any of that — admit only as many turns as the token budget can carry."*

In [ ]:
import sys, asyncio, inspect, random
sys.path[:0] = [".", ".."]
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from nbutil import todo, check, acheck, latency_cdfs, timeline
from scalelab.clock import CLOCK
%matplotlib inline
pd.set_option("display.width", 140)
CLOCK.reset(0.02)   # 50x faster than real time

## 1. The shared pool

A sliding 60-second window of tokens. Above 75 % utilisation latency inflates; above 100 % the pool answers 429 with a retry hint.

In [ ]:
from scalelab.model import SharedPool
from scalelab.resilience import RateLimited, TokenBucket, backoff, CircuitBreaker, call_with_retries
pool = SharedPool(tpm=100_000)
admitted = 0
try:
    while True:
        pool.admit(5_000); admitted += 1
except RateLimited as e:
    print(f"{admitted} calls of 5k tokens admitted, then 429 with retry_after={e.retry_after:.1f}s; utilisation {pool.utilisation:.0%}")

## 2. Smooth your own traffic

A token bucket refills at the rate you chose and tolerates a burst worth `capacity`. Thirty calls arriving at once leave at a steady pace instead of hitting the pool together.

In [ ]:
async def burst(bucket):
    t = []
    for _ in range(30):
        await bucket.acquire(5_000); t.append(CLOCK.now())
    return np.array(t) - t[0]
CLOCK.reset(0.02)
smooth = await burst(TokenBucket(rate=50_000 / 60, capacity=50_000 / 60 * 6))    # 50k TPM, 6-second burst
fig, ax = plt.subplots(figsize=(7, 2.5))
ax.eventplot([np.zeros(30), smooth], lineoffsets=[1, 0], colors=["tab:red", "tab:blue"])
ax.set_yticks([1, 0]); ax.set_yticklabels(["unsmoothed burst", "through the bucket"]); ax.set_xlabel("virtual seconds"); ax.set_title("30 calls of 5k tokens")
plt.tight_layout()
print(f"the burst is spread over {smooth[-1]:.1f} s at the sustained rate of the bucket")

## 3. Jitter

When the pool throttles everyone at once, everyone retries at once — unless the backoff is jittered. Two hundred clients, each retrying until a pool with room for 40 calls per second admits them.

In [ ]:
async def retry_storm(jitter, clients=200, rate=40):
    CLOCK.reset(0.02)
    pool = SharedPool(tpm=rate * 60 * 1000)          # 1k tokens per call, `rate` calls/s
    done, extra_429 = [], 0
    async def client(i):
        nonlocal extra_429
        rng = random.Random(i)
        for attempt in range(1, 8):
            try:
                pool.admit(1000); done.append(CLOCK.now()); return
            except RateLimited:
                if attempt > 1: extra_429 += 1
                await CLOCK.sleep(backoff(attempt, jitter=jitter, rng=rng))
    await asyncio.gather(*(client(i) for i in range(clients)))
    return np.array(done), extra_429

fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
for ax, jitter in zip(axes, (False, True)):
    t, extra = await retry_storm(jitter)
    ax.hist(t, bins=40); ax.set_title(f"{'full jitter' if jitter else 'no jitter'}: {len(t)} served by {t.max():.1f}s, {extra} secondary 429s"); ax.set_xlabel("virtual seconds")
plt.tight_layout()

## 4. The circuit breaker

After enough failures in a window it opens (fail fast for a cooldown), then half-opens for a single probe. A breaker per model lets the gateway move to a sibling model whose pool is not contended.

In [ ]:
CLOCK.reset(0.02)
cb = CircuitBreaker(threshold=3, min_calls=4, ratio=0.5, cooldown=5.0)
for ok in (True, False, False, False):
    cb.record(ok); print(f"record({ok}) -> {cb.state}")
await CLOCK.sleep(5.1); print("after cooldown ->", cb.state)
cb.record(True); print("probe succeeded ->", cb.state)

## 5. Admission control: the brake

Signals → level. Levels 1–2 buy capacity by giving something up (cheaper model, shorter answers, no writes); level 3
sheds with a `Retry-After`. Levels 1–2 are held for a dwell time so the system does not flap; level 3 follows the
instantaneous cap.

In [ ]:
from scalelab.admission import AdmissionController, AdmissionConfig
CLOCK.reset(0.02)
ac = AdmissionController(AdmissionConfig(max_inflight=10, dwell_s=0))
rows = []
for inflight, ratio in [(0, 0.0), (8, 0.0), (0, 0.06), (0, 0.2), (10, 0.0)]:
    ac.inflight = inflight; ac._recent.clear()
    for _ in range(20): ac.note_model_call(rate_limited=random.random() < ratio)
    rows.append({"inflight": inflight, "429 ratio": ratio, "level": ac.compute_level()})
pd.DataFrame(rows)

## Your turn

Fill in each function (replace the `todo()` call), then run the check cell below it. The practice notebook runs end to end with the exercises untouched; the solutions notebook has every check passing.

#### (a) Backoff with full jitter

min(cap, base·2^(attempt−1)), then a uniform draw between 0 and that.

In [ ]:
def my_backoff(attempt, base=0.5, cap=8.0, rng=random):
    return todo()

In [ ]:
def _a():
    rng = random.Random(3)
    draws = [my_backoff(a, rng=rng) for a in range(1, 12) for _ in range(50)]
    assert all(0 <= d <= 8.0 for d in draws)
    assert max(my_backoff(3, rng=random.Random(k)) for k in range(200)) <= 2.0
    assert max(my_backoff(6, rng=random.Random(k)) for k in range(200)) > 6.0
check("a: backoff", _a)

#### (b) A token bucket

Implement `try_acquire`: refill by elapsed time × rate (capped), then either take the tokens (return 0) or return the seconds to wait.

In [ ]:
class MiniBucket:
    def __init__(self, rate, capacity):
        self.rate, self.capacity, self.tokens, self.last = rate, capacity, capacity, CLOCK.now()
    def try_acquire(self, n=1.0):
        return todo()

In [ ]:
def _b():
    CLOCK.reset(0.02)
    b = MiniBucket(rate=10, capacity=10)
    assert all(b.try_acquire(1) == 0 for _ in range(10))
    w = b.try_acquire(5)
    assert 0.3 < w <= 0.5, w
check("b: token bucket", _b)

#### (c) Degrade level

Reproduce the level function without hysteresis: 1 above 80 % of the cap or ≥ 5 % 429s or queue age ≥ 10 s; 2 at ≥ 15 % 429s; 3 at the cap or queue age ≥ 30 s.

In [ ]:
def degrade_level(inflight, cap, ratio_429, queue_age_s):
    return todo()

In [ ]:
def _c():
    assert degrade_level(10, 100, 0, 0) == 0
    assert degrade_level(85, 100, 0, 0) == 1 and degrade_level(0, 100, 0.06, 0) == 1 and degrade_level(0, 100, 0, 12) == 1
    assert degrade_level(0, 100, 0.2, 0) == 2
    assert degrade_level(100, 100, 0, 0) == 3 and degrade_level(0, 100, 0, 31) == 3
check("c: degrade level", _c)

#### (d) Shedding decision

Admit if the level is below 3 or the caller is priority (≤ 2). A shed reply carries a Retry-After that grows with the level, and the *client* must jitter it (return the jittered value for a given rng).

In [ ]:
def shed_decision(level, priority, rng=random):
    """-> (admitted: bool, retry_after: float | None)"""
    return todo()

In [ ]:
def _d():
    assert shed_decision(2, 5) == (True, None)
    assert shed_decision(3, 1) == (True, None)
    ok, ra = shed_decision(3, 5, random.Random(1))
    assert not ok and 10 <= ra <= 30
check("d: shedding", _d)

## Takeaways

- 429 = contention on a shared pool. Smooth within the minute; back off with full jitter; never retry past the deadline.
- A breaker per model, and a sibling model with its own pool as the fallback.
- Admission control at the edge with degrade levels: cheaper first, shed last, priorities bypass the cap, hysteresis on the degrade levels.
- The in-flight cap starts from the token budget (notebook 01) and is tuned from load tests (notebook 04).

## Verify before relying on it

Google's documented reaction to 429s (backoff, global endpoint, smoothing); PT request-type headers (`dedicated` / `shared`, spill tier `priority` / `flex`); whether the SDK retries by default (it does not in google-genai 2.x).